# plot_02. Sea ice: figures

Everything here is drawn from what notebook calc_04 saved, so nothing slow runs. calc_04 explains each step and has a quick look at each; the figures of its methods step by step (how a field goes onto the 2.5° grid, how the ice edge becomes a latitude) are there too, beside their calculations. Each section here first works out what its figures need from the saves (**Calculations**), then draws one model (**Sample plots**) and every model (**Figures**). **Which results** sets the folder the results are opened from.

Every map figure is built the same way, with `extant/plots/antarctica.py`: `map_figure_size` gives the figure's size and `map_axes` its maps and colour bars; then each layer is drawn on those axes by its own call (`heatmaps_of_antarctica` for a field, `ice_edges` for an ice edge), and the rows and columns are labelled. A row of a different kind is a slice of the same axes (`axes[2]`) with its own calls. Every other figure gets its size from `charts.chart_figure_size` and its panels from a GridSpec (`fig.add_gridspec(...).subplots()`, in a figure made with `layout='constrained'`, with `fig.legend` and `fig.colorbar` under them), then each layer is drawn on them (`charts.lines_grid`, `charts.model_dots_grid`, a panel of `sip`, or plain `ax.plot`), and the panels are labelled.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from extant import sea_ice
from extant.config import FORCED_EXPERIMENTS, SATELLITE_YEAR_RANGE, WINDOW
from extant.utils.datatree import tree_to_dataset
from extant.utils.logger import setup_logging
from extant.plots import antarctica, charts, era5_evaluation as pe, forced_response as fr, paper_panels, sea_ice as sip
from plotting_modules.constants import FORCING_COLORS, SEASONS
from plotting_modules import plot_constants

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [2]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`FORCED_EXPERIMENTS`, `SATELLITE_YEAR_RANGE`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [ ]:
#(t): Where calc_04 and notebooks calc_02a to calc_02c saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')

In [3]:
#(t): The model and experiment of the one-model figures
EXAMPLE_MODEL = 'CanESM5'
EXPERIMENT = 'hist-GHG'

### Colour scales

One per quantity, so a quantity looks the same in every figure. Each is the colour settings of `antarctica.heatmaps_of_antarctica`: the edges of the colour steps (`levels`), the colours, which ends get a colour of their own for values beyond the levels (`extend`), the colour bar's ticks and label; a figure passes one as `**concentration_colours`. A difference in concentration is blue where there is more ice and red where there is less, so that ice lost reads like warming on the temperature maps.

In [ ]:
#(t): The concentration (%), light to dark blue
concentration_colours = dict(levels=np.arange(0, 101, 10), cmap='Blues', extend='neither',
                             label='Sea-ice concentration (%)')

#(t): A difference in concentration (%): blue where there is more ice, red where there is less
concentration_change_colours = dict(levels=np.arange(-55, 56, 10), cmap='RdBu', extend='both',
                                    ticks=np.arange(-50, 51, 10), label='Difference in sea-ice concentration (%)')

#(t): The correlation of the sea ice and the temperature
correlation_colours = dict(levels=np.round(np.arange(-1, 1.01, 0.2), 1), cmap='PuOr_r', extend='neither',
                           label='Correlation of sea ice and temperature')

#(t): The temperature's mean change and change in width, on notebook plot_03's scales (fr.colour_scales; ranges in fr.RANGES)
tas_scales = fr.colour_scales('tas')
mean_change_colours = dict(levels=tas_scales.change.levels, cmap=tas_scales.change.cmap, extend='both',
                           label=tas_scales.change.label)
width_change_colours = dict(levels=tas_scales.spread_change.levels, cmap=tas_scales.spread_change.cmap, extend='both',
                            label=tas_scales.spread_change.label)

#(t): ERA5's ice edge: black, as ERA5 is in every figure, and dotted (its line plots have dots), apart from the
#(t): models' solid edges
era5_edge_style = dict(color=charts.ERA5_LINE['color'], linestyle=':', linewidth=1.6)

## Which Results

Run **one** of the next two sections: each sets the folders the results are opened from.

### Sample data

In [ ]:
#(t): The sample's results: calc_04's, and notebooks calc_02a to calc_02c's temperature results
RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'sample'
TAS_RESULTS_DIR = SCRATCH_DIR / 'results' / 'tas' / 'sample'

### Full data

In [ ]:
#(t): The full data's results: calc_04's, and notebooks calc_02a to calc_02c's temperature results
RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'full'
TAS_RESULTS_DIR = SCRATCH_DIR / 'results' / 'tas' / 'full'

## Open the Results

First what calc_04 saved in `RESULTS_DIR`, then each result opened.

In [ ]:
#(t): What calc_04 saved in RESULTS_DIR
RESULTS_DIR, sorted(path.name for path in RESULTS_DIR.glob('*.zarr'))

In [ ]:
#(t): Each member's seasonal extent and its model's continent (km²), /<model>/<experiment>
ice_extents_tree = xr.open_datatree(RESULTS_DIR / 'extent.zarr', engine='zarr').load()

In [ ]:
#(t): The ensemble-mean concentration over each experiment's final WINDOW years (%)
ice_climatology_da = xr.open_zarr(RESULTS_DIR / 'climatology.zarr')['siconc'].load()

In [ ]:
#(t): Each historical member's seasonal cycle of extent over SATELLITE_YEAR_RANGE (million km²), /<model>/historical
historical_cycle_tree = xr.open_datatree(RESULTS_DIR / 'historical_cycle.zarr', engine='zarr').load()

In [ ]:
#(t): ... and ERA5's
era5_cycle_da = xr.open_zarr(RESULTS_DIR / 'era5_cycle.zarr')['extent'].load()

In [ ]:
#(t): Each model's historical concentration over SATELLITE_YEAR_RANGE (%)
historical_climatology_da = xr.open_zarr(RESULTS_DIR / 'historical_climatology.zarr')['siconc'].load()

In [ ]:
#(t): ... and ERA5's
era5_climatology_da = xr.open_zarr(RESULTS_DIR / 'era5_climatology.zarr')['siconc'].load()

In [ ]:
#(t): Each historical member's ice edge over SATELLITE_YEAR_RANGE, as an equivalent latitude (degrees),
#(t): /<model>/historical
historical_member_edges_tree = xr.open_datatree(RESULTS_DIR / 'historical_member_edges.zarr', engine='zarr').load()

In [ ]:
#(t): ... and ERA5's
era5_edge_da = xr.open_zarr(RESULTS_DIR / 'era5_edge.zarr')['edge_latitude'].load()

In [ ]:
#(t): ERA5 among each model's members, /<model>/historical, on (statistic, season, member)
era5_extent_test_tree = xr.open_datatree(RESULTS_DIR / 'era5_extent_test.zarr', engine='zarr').load()

In [ ]:
#(t): Each experiment's change in concentration against hist-nat (%)
ice_change_da = xr.open_zarr(RESULTS_DIR / 'ice_change.zarr')['siconc'].load()

In [ ]:
#(t): ... and in extent (million km²) and edge (degrees)
extent_change_ds = xr.open_zarr(RESULTS_DIR / 'extent_change.zarr').load()

In [ ]:
#(t): Each cell's zone (the numbers of sea_ice.ZONES)
ice_zones_da = xr.open_zarr(RESULTS_DIR / 'zones.zarr')['zone'].load()

In [ ]:
#(t): The temperature changes in each zone, and each zone's area
tas_zone_means_ds = xr.open_zarr(RESULTS_DIR / 'zone_means.zarr').load()

In [ ]:
#(t): The temperature changes over hist-nat's ice
ice_region_change_ds = xr.open_zarr(RESULTS_DIR / 'ice_region_change.zarr').load()

In [ ]:
#(t): Notebook calc_02b's mean change, for the figures of what goes into the zones
mean_change_da = xr.open_zarr(TAS_RESULTS_DIR / 'mean_bootstrap.zarr')['mean_change'].load()

In [ ]:
#(t): ... and its change in width, Q95 - Q05
width_change_da = xr.open_zarr(TAS_RESULTS_DIR / 'qrange_change.zarr')['qrange_change'].load()

## 1. Extent

Each experiment's extent through time: the ensemble mean and the members' 5–95%. How the ice edge becomes an equivalent latitude is drawn step by step in calc_04, section 4.

### Calculations

In [ ]:
#(t): Each experiment's ensemble-mean extent, every year and season (km²)
ice_extent_mean_tree = ice_extents_tree.mean('member')

In [ ]:
#(t): ... and the members' 5% and 95% (km²)
ice_extent_spread_tree = xr.DataTree.from_dict({node.path: node['extent'].quantile([0.05, 0.95], 'member').to_dataset()
                                                for node in ice_extents_tree.leaves})

In [ ]:
#(t): Every model and experiment side by side, on (model, experiment, year, season)
ice_extent_mean_ds = tree_to_dataset(ice_extent_mean_tree, dims=['model', 'experiment'])
ice_extent_spread_ds = tree_to_dataset(ice_extent_spread_tree, dims=['model', 'experiment'])

In [ ]:
#(t): Together, for the figure: mean, low (5%) and high (95%)
ice_extent_summary_ds = xr.Dataset({'mean': ice_extent_mean_ds['extent'],
                                    'low': ice_extent_spread_ds['extent'].sel(quantile=0.05, drop=True),
                                    'high': ice_extent_spread_ds['extent'].sel(quantile=0.95, drop=True)})
ice_extent_summary_ds

### Figures

In [ ]:
#(t): Each experiment's extent through time (million km²): its ensemble mean (line) and its members' 5-95% (band),
#(t): seasons down and models across; the y axis shared along each season's row
extent_experiments = ['hist-nat'] + FORCED_EXPERIMENTS
extent_models = list(ice_extent_summary_ds['model'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(SEASONS), n_cols=len(extent_models), n_legends=1),
                 layout='constrained')
axes = fig.add_gridspec(len(SEASONS), len(extent_models)).subplots(sharex=True, sharey='row', squeeze=False)

for experiment in extent_experiments:
    experiment_extent_ds = ice_extent_summary_ds.sel(experiment=experiment, season=SEASONS) / 1e6
    charts.bands_grid(experiment_extent_ds['low'], experiment_extent_ds['high'], 'year', 'season', 'model', axes=axes,
                      color=FORCING_COLORS[experiment], alpha=0.15)
    charts.lines_grid(experiment_extent_ds['mean'], 'year', 'season', 'model', axes=axes,
                      color=FORCING_COLORS[experiment], linewidth=1.4)

for ax, model in zip(axes[0], extent_models):
    ax.set_title(model, loc='left')
for ax, season in zip(axes[:, 0], SEASONS):
    ax.set_ylabel(f'{season}\nExtent (million km²)')
fig.legend(handles=charts.forcing_legend_handles(extent_experiments)
           + [plot_constants.members_range_patch],
           loc='outside lower center', ncols=len(extent_experiments) + 1, frameon=False)
fig.suptitle('Antarctic sea-ice extent (cells with at least 15% ice), every model, experiment and season', x=0.01,
             ha='left');

## 2. Comparison with ERA5

Each model's historical members against ERA5 over `SATELLITE_YEAR_RANGE`, counted on the same grid in the same way. The usual CMIP6 picture (Roach et al., 2020): most models have too little ice in summer and a spread of winter extents round the observed one.

### Calculations

**Concentration and edge**

In [ ]:
#(t): The difference: each model's historical ensemble-mean concentration minus ERA5's (%)
historical_bias_da = historical_climatology_da - era5_climatology_da

In [ ]:
#(t): ERA5's concentration on every model's row, so its edge can be drawn on each (the same on every row)
era5_every_row_da = era5_climatology_da.broadcast_like(historical_climatology_da)

**ERA5 among the members**

In [33]:
#(t): Every model's verdicts, on (model, statistic, season)
era5_extent_verdicts_ds = xr.concat([node.to_dataset()[['percentile', 'verdict', 'testable']]
                                     for node in era5_extent_test_tree.leaves],
                                    dim=xr.Variable('model', [node.parent.name for node in era5_extent_test_tree.leaves]))
era5_extent_verdicts_ds

<xarray.Dataset> Size: 2kB
Dimensions:     (model: 6, statistic: 3, season: 4)
Coordinates:
  * statistic   (statistic) object 24B 'mean' 'std' 'trend'
  * season      (season) object 32B 'DJF' 'JJA' 'MAM' 'SON'
  * model       (model) <U15 360B 'GISS-E2-1-G' ... 'MPI-ESM1-2-LR'
Data variables:
    percentile  (model, statistic, season) float64 576B 100.0 100.0 ... 0.0
    verdict     (model, statistic, season) float64 576B 0.0 0.0 0.0 ... 0.0 0.0
    testable    (model, statistic, season) float64 576B 0.0 0.0 0.0 ... 0.0 0.0

### Sample plots

In [ ]:
#(t): Quick look: ERA5's concentration in DJF
era5_climatology_da.sel(season='DJF').plot();

In [ ]:
#(t): Quick look: ERA5's field on the first model's row, in DJF (the same on every row)
era5_every_row_da.sel(season='DJF').isel(model=0).plot();

### Figures

**Seasonal cycle**

In [ ]:
#(t): Each model's seasonal cycle against ERA5's: every member (light), their mean (dark) and ERA5 (black), two rows
#(t): of three models
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=2, n_cols=3, n_legends=1), layout='constrained')
#(c): Two rows of three panels, sharing their x and y axes
axes = fig.add_gridspec(2, 3).subplots(sharex=True, sharey=True)

#(c): 1. The models, one panel each: every member, then their mean
charts.lines_grid(historical_cycle_tree, 'month', axes=axes, variable='extent', **charts.MEMBER_LINE)
charts.lines_grid(historical_cycle_tree.mean('member'), 'month', axes=axes, variable='extent',
                  **charts.ENSEMBLE_MEAN_LINE)

#(c): 2. ERA5 on every panel
for ax in axes.ravel():
    charts.plot_lines(era5_cycle_da, 'month', ax=ax, **charts.ERA5_LINE)

#(c): The axes are shared, so the months set on one panel are on all
axes[0, 0].set_xticks(range(1, 13), list('JFMAMJJASOND'))
for ax in axes[:, 0]:
    ax.set_ylabel('Extent (million km²)')
fig.legend(handles=[Line2D([], [], **charts.MEMBER_LINE), Line2D([], [], **charts.ENSEMBLE_MEAN_LINE),
                    Line2D([], [], **charts.ERA5_LINE)], loc='outside lower center', ncols=3, frameon=False)
fig.suptitle(f'Seasonal cycle of sea-ice extent, historical against ERA5, '
             f'{SATELLITE_YEAR_RANGE.start}-{SATELLITE_YEAR_RANGE.stop}', x=0.01, ha='left');

**Concentration and edge**

The historical ensemble-mean concentration minus ERA5's, with both edges in black: the model's solid and ERA5's dotted. Blue: the model has more ice.

In [ ]:
#(t): Quick look: every model's historical edge (colours) and ERA5's (black) in DJF, on one map
colors = ['red', 'green', 'blue', 'brown', 'yellow', 'orange']
fig, ax = plt.subplots(subplot_kw=dict(projection=ccrs.SouthPolarStereo()))
antarctica.style_antarctic_ax(ax)
for i, model in enumerate(historical_climatology_da.model.values):
    antarctica.ice_edge(historical_climatology_da.sel(model=model, season='DJF'), ax=ax, color=colors[i])
antarctica.ice_edge(era5_climatology_da.sel(season='DJF'), ax=ax, color='black')
#(c): The legend by hand: the edges are contour lines, which carry no labels
legend_lines = [plt.Line2D([], [], color=colors[i], label=model) for i, model in enumerate(historical_climatology_da.model.values)]
ax.legend(handles=legend_lines + [plt.Line2D([], [], color='black', label='ERA5')], fontsize=7, loc='center left',
          bbox_to_anchor=(1, 0.5));

In [ ]:
#(t): Every model (rows) and season (columns): the difference, each model's own edge (solid) and ERA5's (dotted)
n_models = historical_bias_da.sizes['model']
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=n_models, n_cols=len(SEASONS), n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=n_models, n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(historical_bias_da.sel(season=SEASONS), 'model', 'season', axes=axes,
                                  cax=cax, **concentration_change_colours)
antarctica.ice_edges(historical_climatology_da.sel(season=SEASONS), 'model', 'season', axes=axes)
antarctica.ice_edges(era5_every_row_da.sel(season=SEASONS), 'model', 'season', axes=axes, **era5_edge_style)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, historical_bias_da['model'].values)
fig.legend(handles=[plt.Line2D([], [], color='k', label="The model's edge"),
                    plt.Line2D([], [], **era5_edge_style,
                               label="ERA5's edge")], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'Sea-ice concentration, historical minus ERA5, {SATELLITE_YEAR_RANGE.start}-{SATELLITE_YEAR_RANGE.stop}');

**Edge as an equivalent latitude**

In [ ]:
#(t): Where each model's ice edge is, season by season: its members (dots) and their mean (marker), against ERA5's
#(t): (black bar)
edge_models = [node.parent.name for node in historical_member_edges_tree.match('*/historical').leaves]
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=1, panel_width=7.5, panel_height=3.4),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
sip.plot_edge_latitudes(historical_member_edges_tree, era5_edge_da, SEASONS, ax=axes[0, 0])
axes[0, 0].set_title("Markers: each model's mean; dots: its members", loc='left')
axes[0, 0].set_ylabel('Equivalent latitude of the ice edge')
fig.legend(handles=charts.model_legend_handles(edge_models, experiment='historical')
           + [Line2D([], [], color=charts.ERA5_LINE['color'], linewidth=2.2, label='ERA5')],
           loc='outside lower center', ncols=len(edge_models) + 1, frameon=False)
fig.suptitle(f'Where the ice edge is: historical against ERA5, '
             f'{SATELLITE_YEAR_RANGE.start}-{SATELLITE_YEAR_RANGE.stop}', x=0.01, ha='left');

**ERA5 among the members**

calc_04, section 6: the extent's mean, variability and trend. A tick: ERA5 inside the members' 5–95%; a cross: outside (and what that means for the model); a dash: too few members to flag anything.

In [ ]:
#(t): One figure per statistic of the extent: each model's members (dots, their 5-95% shaded, their median a line)
#(t): and ERA5 (black), models down and seasons across, each panel with its verdict
era5_test_leaves = era5_extent_test_tree.match('*/historical').leaves
for statistic in ['mean', 'std', 'trend']:
    diagnostic = sea_ice.ERA5_DIAGNOSTICS[statistic]
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(era5_test_leaves), n_cols=len(SEASONS), n_legends=1,
                                                      panel_width=2.3, panel_height=0.75, left_margin=1.5),
                     layout='constrained')
    axes = fig.add_gridspec(len(era5_test_leaves), len(SEASONS)).subplots(squeeze=False)
    for i, node in enumerate(era5_test_leaves):
        for j, season in enumerate(SEASONS):
            cell_ds = node.to_dataset().sel(statistic=statistic, season=season)
            ok, message = sip.era5_verdict(cell_ds, diagnostic)
            if message == 'no data':
                axes[i, j].set_axis_off()
                continue
            pe.plot_member_strip(cell_ds.dropna('member', how='all'), ax=axes[i, j])
            pe.status_line(axes[i, j], 0.01, 0.99, ok, message, fontsize=7)
        axes[i, 0].set_ylabel(node.parent.name, rotation=0, ha='right', va='center', fontweight='bold')

    for ax, season in zip(axes[0], SEASONS):
        ax.set_title(season, loc='left', pad=13)
    for ax in axes[-1]:
        ax.set_xlabel(diagnostic.units)
    fig.legend(handles=[
Line2D([], [], linestyle='none', marker='o', markersize=5, color=pe.ENSEMBLE['inner'], markeredgecolor='white',
    label='Each member'),
Patch(color=pe.ENSEMBLE['range'], label="Members' 5–95%"),
Line2D([], [], color=pe.ENSEMBLE['median'], linewidth=1, label='Member median'),
Line2D([], [], color=pe.ERA5_COLOR, linewidth=1.8, marker='D', markersize=5, markeredgecolor='white',
    label='ERA5'),
], loc='outside lower center', ncols=4, frameon=False)
    fig.suptitle(f"Sea-ice extent, {diagnostic.label.lower()}: ERA5 among each model's members", x=0.01, ha='left')

In [ ]:
#(t): Every model (rows), statistic and season (columns): ERA5's percentile among the members, outlined where it is
#(t): significant
n_scorecard_columns = era5_extent_verdicts_ds.sizes['statistic'] * len(SEASONS)
n_scorecard_rows = era5_extent_verdicts_ds.sizes['model']
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, n_colorbars=1,
                                                  panel_width=0.44 * n_scorecard_columns,
                                                  panel_height=0.46 * n_scorecard_rows, left_margin=1.5),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
scorecard_image = pe.plot_scorecard(era5_extent_verdicts_ds.sel(season=SEASONS), ax=axes[0, 0])
fig.colorbar(scorecard_image, ax=axes, location='bottom', shrink=0.6, ticks=pe.PERCENTILE_BOUNDS,
             label='ERA5 percentile among the members')
fig.suptitle(f"Sea-ice extent: where ERA5 falls among each model's historical members, "
             f"{SATELLITE_YEAR_RANGE.start}-{SATELLITE_YEAR_RANGE.stop}", x=0.01, ha='left');

## 3. Sea Ice and Temperature Change

The change in sea ice against hist-nat, the zones where it changed and the temperature changes in each, and the change in extent against the change in temperature (calc_04, section 7).

### Calculations

In [ ]:
#(t): EXAMPLE_MODEL's concentration over the final WINDOW years, every experiment and season
model_ice_da = ice_climatology_da.sel(model=EXAMPLE_MODEL, season=SEASONS)
model_ice_da

In [ ]:
#(t): The multi-model median of the change, and of the concentration (for the edges)
median_ice_change_da = ice_change_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS).median('model')
median_ice_da = ice_climatology_da.sel(season=SEASONS).median('model')
median_ice_change_da

### Sample plots

**Sea-ice change**

In [ ]:
#(t): What goes in: hist-nat's and EXPERIMENT's concentration, each with its edge (one colour bar), then the change
#(t): with both edges
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
#(c): The concentration's colour bar under rows 1 and 2, the change's under row 3
concentration_cax = antarctica.colorbar_axis_below(axes[:2])
change_cax = antarctica.colorbar_axis_below(axes[2])

#(c): Rows 1 and 2: hist-nat's and EXPERIMENT's concentration, each with its own edge
antarctica.heatmaps_of_antarctica(model_ice_da.sel(experiment=['hist-nat', EXPERIMENT]), 'experiment', 'season',
                                  axes=axes[:2], cax=concentration_cax, **concentration_colours)
antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat'), None, 'season', axes=axes[0], linestyle='dashed')
antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT), None, 'season', axes=axes[1])

#(c): Row 3: the change, with both edges
antarctica.heatmaps_of_antarctica(ice_change_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS), None,
                                  'season',
                                  axes=axes[2], cax=change_cax, **concentration_change_colours)
antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat'), None, 'season', axes=axes[2], linestyle='dashed')
antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT), None, 'season', axes=axes[2])

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [f'hist-nat\nfinal {WINDOW} years', f'{EXPERIMENT}\nfinal {WINDOW} years',
                             f'Change\n{EXPERIMENT} − hist-nat'])
fig.legend(handles=[plot_constants.hist_nat_edge_handle,
                    plt.Line2D([], [], color='k',
                               label=f"{EXPERIMENT}'s ice edge")], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'What goes into the change in sea ice: {EXAMPLE_MODEL}, {EXPERIMENT}');

In [ ]:
#(t): One model: EXAMPLE_MODEL's change in every experiment (rows), with hist-nat's edge (dashed) and each experiment's
#(t): (solid)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(ice_change_da.sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS,
                                                    season=SEASONS), 'experiment',
                                  'season', axes=axes, cax=cax, **concentration_change_colours)
#(c): hist-nat's edge on every row: given an experiment dim with the rows' experiments, so each row finds it
antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                     'experiment', 'season', axes=axes, linestyle='dashed')
antarctica.ice_edges(model_ice_da.sel(experiment=FORCED_EXPERIMENTS), 'experiment', 'season', axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[plot_constants.hist_nat_edge_handle,
                    plot_constants.experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'{EXAMPLE_MODEL}: change in sea-ice concentration, final {WINDOW} years against hist-nat');

**Zones**

In [ ]:
#(t): What goes in: EXAMPLE_MODEL and EXPERIMENT's zones, and the mean and width changes behind them, each with both
#(t): edges
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=3, n_legends=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
#(c): Each row's colour bar under it
zone_cax = antarctica.colorbar_axis_below(axes[0])
mean_cax = antarctica.colorbar_axis_below(axes[1])
width_cax = antarctica.colorbar_axis_below(axes[2])

#(c): One row each, with its own colour bar; the zones' bar names its classes
zone_cbar = antarctica.heatmaps_of_antarctica(ice_zones_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                               season=SEASONS),
                                              None, 'season', axes=axes[0], cax=zone_cax,
                                              **plot_constants.zone_scale.colours())
zone_cbar.set_ticklabels(sea_ice.ZONES)
antarctica.heatmaps_of_antarctica(mean_change_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS), None,
                                  'season', axes=axes[1], cax=mean_cax, **mean_change_colours)
antarctica.heatmaps_of_antarctica(width_change_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS), None,
                                  'season', axes=axes[2], cax=width_cax, **width_change_colours)

#(c): Both edges on every row: hist-nat's (dashed) and EXPERIMENT's (solid)
for row_axes in axes:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat'), None, 'season', axes=row_axes, linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT), None, 'season', axes=row_axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, ['Zone', f'Δ mean\n{EXPERIMENT} − hist-nat', f'Δ(Q95 − Q05)\n{EXPERIMENT} − hist-nat'])
fig.legend(handles=[plot_constants.hist_nat_edge_handle,
                    plt.Line2D([], [], color='k',
                               label=f"{EXPERIMENT}'s ice edge")], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'Where the ice changed, and the temperature: {EXAMPLE_MODEL}, {EXPERIMENT}, final {WINDOW} years');

### Figures

**Sea-ice change**

Each experiment's ensemble-mean concentration over the final `WINDOW` years minus hist-nat's, with hist-nat's edge (dashed) and the experiment's (solid). Blue: more ice; red: less.

In [ ]:
#(t): Every model (rows), one figure per experiment, with hist-nat's edge (dashed) and the experiment's (solid)
n_models = ice_change_da.sizes['model']
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=n_models, n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=n_models, n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(ice_change_da.sel(experiment=experiment, season=SEASONS), 'model', 'season',
                                      axes=axes, cax=cax, **concentration_change_colours)
    antarctica.ice_edges(ice_climatology_da.sel(experiment='hist-nat', season=SEASONS), 'model', 'season', axes=axes,
                         linestyle='dashed')
    antarctica.ice_edges(ice_climatology_da.sel(experiment=experiment, season=SEASONS), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, ice_change_da['model'].values)
    fig.legend(handles=[plot_constants.hist_nat_edge_handle,
                        plt.Line2D([], [], color='k',
                                   label=f"{experiment}'s ice edge")], loc='outside lower center', ncols=2, fontsize=8)
    fig.suptitle(f'Change in sea-ice concentration, final {WINDOW} years against hist-nat: {experiment}')

In [ ]:
#(t): The multi-model median, every experiment (rows), with the edges of the models' median concentration
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(median_ice_change_da, 'experiment', 'season', axes=axes, cax=cax,
                                  **concentration_change_colours)
antarctica.ice_edges(median_ice_da.sel(experiment='hist-nat', drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                     'experiment', 'season', axes=axes, linestyle='dashed')
antarctica.ice_edges(median_ice_da.sel(experiment=FORCED_EXPERIMENTS), 'experiment', 'season', axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[plot_constants.hist_nat_edge_handle,
                    plot_constants.experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'Change in sea-ice concentration, final {WINDOW} years against hist-nat, multi-model median');

**Zones**

Each ocean cell sorted by its concentration in hist-nat's final years and the experiment's: open water in both, ice in both, ice lost, ice gained, or the continent. If the expectation holds, "ice lost" warms more than "open water" and "ice in both" in autumn and winter, and narrows.

In [ ]:
#(t): The legend of the zone figures below: each model's marker, and the bar of their median
zone_legend_handles = (charts.model_legend_handles(tas_zone_means_ds['model'].values)
                       + [plot_constants.multi_model_median_patch])

In [ ]:
#(t): The change in the mean temperature in each zone: experiments down, seasons across; a bar for the median of the
#(t): models and a marker for each model, in the experiment's colour
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(tas_zone_means_ds['mean'].sel(experiment=experiment, season=SEASONS), 'zone', None,
                           'season', axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ mean (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle(f'Change in the mean temperature in each zone, final {WINDOW} years against hist-nat', x=0.01, ha='left');

In [ ]:
#(t): ... the change in the width, Q95 − Q05
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(tas_zone_means_ds['width'].sel(experiment=experiment, season=SEASONS), 'zone', None,
                           'season', axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ(Q95 − Q05) (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle(f'Change in the width in each zone, final {WINDOW} years against hist-nat', x=0.01, ha='left');

In [ ]:
#(t): ... the change in the lower (cold) tail, Q50 − Q05: negative where the cold extremes warm more than the median
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(tas_zone_means_ds['lower tail'].sel(experiment=experiment, season=SEASONS), 'zone', None, 'season',
                          axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ(Q50 − Q05) (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle('Change in the lower (cold) tail in each zone: negative where the cold extremes warm more than the median',
             x=0.01, ha='left');

In [ ]:
#(t): ... the change in the upper (warm) tail, Q95 − Q50
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(tas_zone_means_ds['upper tail'].sel(experiment=experiment, season=SEASONS), 'zone', None, 'season',
                          axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ(Q95 − Q50) (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle('Change in the upper (warm) tail in each zone', x=0.01, ha='left');

**Extent change against temperature change**

One point per model and forcing: the change in extent against the change in the mean, and in the width, averaged over the cells with ice in hist-nat. If losing ice drives them, more loss (further left) goes with more warming and more narrowing.

In [ ]:
#(t): The legends of the two figures below: each forcing's colour, and each model's marker
forcing_legend_handles = charts.forcing_legend_handles(FORCED_EXPERIMENTS)
model_legend_handles = charts.model_legend_handles(extent_change_ds['model'].values)

In [ ]:
#(t): The change in extent against the change in the mean over hist-nat's ice: one point per model (marker) and forcing
#(t): (colour), one panel per season; r is the correlation over all the points
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    charts.plot_scatter_by_forcing(extent_change_ds['extent_change'].sel(experiment=FORCED_EXPERIMENTS, season=season),
                                         ice_region_change_ds['mean'].sel(experiment=FORCED_EXPERIMENTS,
                                                                          season=season), ax=ax)
    ax.set_title(season, loc='left')
    ax.set_xlabel('Δ extent (million km²)')
axes[0, 0].set_ylabel("Δ mean over hist-nat's ice (°C)")
legend_axes[0].legend(handles=forcing_legend_handles, loc='center', ncols=len(forcing_legend_handles), frameon=False)
legend_axes[1].legend(handles=model_legend_handles, loc='center', ncols=len(model_legend_handles), frameon=False)
fig.suptitle(f"Change in extent against change in the mean temperature over hist-nat's ice, final {WINDOW} years",
             x=0.01, ha='left');

In [ ]:
#(t): ... and against the change in the width over hist-nat's ice
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    charts.plot_scatter_by_forcing(extent_change_ds['extent_change'].sel(experiment=FORCED_EXPERIMENTS, season=season),
                                         ice_region_change_ds['width'].sel(experiment=FORCED_EXPERIMENTS,
                                                                           season=season), ax=ax)
    ax.set_title(season, loc='left')
    ax.set_xlabel('Δ extent (million km²)')
axes[0, 0].set_ylabel("Δ(Q95 − Q05) over hist-nat's ice (°C)")
legend_axes[0].legend(handles=forcing_legend_handles, loc='center', ncols=len(forcing_legend_handles), frameon=False)
legend_axes[1].legend(handles=model_legend_handles, loc='center', ncols=len(model_legend_handles), frameon=False)
fig.suptitle(f"Change in extent against change in the width over hist-nat's ice, final {WINDOW} years", x=0.01,
             ha='left');

### Year-to-year correlation

Strongly negative where a season with more ice is a colder season: that is where the ice sets the year-to-year variability of the temperature. ERA5's in the last row.

#### Calculations

In [ ]:
#(t): The correlations, every model and ERA5
coupling_rows_da = xr.open_zarr(RESULTS_DIR / 'ice_tas_correlation.zarr')['correlation'].load()

In [ ]:
#(t): Each row's own edge: hist-nat's final years for the models, ERA5's 1979-2013, in the rows' order
coupling_ice_da = xr.concat([ice_climatology_da.sel(experiment='hist-nat', drop=True),
                             era5_climatology_da.expand_dims(model=['ERA5'])], dim='model')
coupling_ice_da = coupling_ice_da.sel(model=coupling_rows_da['model'])

#### Figures

In [ ]:
#(t): Every model and ERA5 (rows) and season (columns): the correlation, with each row's own edge
n_rows = coupling_rows_da.sizes['model']
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=n_rows, n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=n_rows, n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(coupling_rows_da.sel(season=SEASONS), 'model', 'season', axes=axes, cax=cax,
                                  **correlation_colours)
antarctica.ice_edges(coupling_ice_da.sel(season=SEASONS), 'model', 'season', axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, coupling_rows_da['model'].values)
fig.suptitle('Year to year, how the sea ice and the temperature move together: hist-nat, and ERA5');